In [ ]:
# import shutil
# import os

# # Delete HateDeRC directory if it exists
# if os.path.exists('HateDeRC'):
#   shutil.rmtree('HateDeRC')
# !git clone https://github.com/jamesalv/HateDeRC
# %cd HateDeRC
# !

In [ ]:
from TrainingConfig import TrainingConfig
import numpy as np
import torch
from transformers import AutoTokenizer
import json

In [ ]:
data_path = 'Data/dataset.json'

In [ ]:
config = TrainingConfig()

config.class_weighting = True
config.learning_rate = 1e-5
config.batch_size = 32
config.num_epochs = 5
config.hidden_dropout_prob = 0.1
config.classification_mode = 'binary'
config.num_labels = 2

# Attention Training Configurations
config.lambda_attn = 0.3           # λ in Eq. 4 (L = α·L_CLS + λ·L_attn); matches the reported results
config.ranking_margin = 0.1        # Minimum margin between token pairs
config.ranking_threshold = 0.05    # Min difference to consider pairs significant

In [ ]:
# --- Main multi-architecture ablation (produces paper Tables 3-11) ---
# Swap this back in to reproduce the main results.
# ablations = [
#   {"model_name": "bert-base-uncased",        "train_attention": False},
#   {"model_name": "bert-base-uncased",        "train_attention": True},
#   {"model_name": "distilbert-base-uncased",  "train_attention": False},
#   {"model_name": "distilbert-base-uncased",  "train_attention": True},
#   {"model_name": "microsoft/deberta-v3-base","train_attention": False},
#   {"model_name": "microsoft/deberta-v3-base","train_attention": True},
# ]

# --- Hyperparameter sensitivity sweep (reviewer revision) ---
# One-at-a-time around the reported config (λ=0.3, m=0.1) on BERT/binary.
# Each dict is applied via `setattr(config, key, value)` in the run loop,
# so any TrainingConfig field (lambda_attn, ranking_margin, ...) works here.
ablations = [
    # shared center: the λ=0.3 point of the λ-sweep AND the m=0.1 point of
    # the m-sweep (aggregation reuses it for both)
    {"model_name": "bert-base-uncased", "train_attention": True,
     "lambda_attn": 0.3, "ranking_margin": 0.1},
]
# λ sweep (m fixed at 0.1); λ=0.0 == no attention supervision (baseline anchor)
for _lv in [0.0, 0.1, 0.5, 1.0, 2.0]:
    ablations.append({"model_name": "bert-base-uncased",
                      "train_attention": _lv > 0,
                      "lambda_attn": _lv, "ranking_margin": 0.1})
# m sweep (λ fixed at 0.3, attention supervision on)
for _mv in [0.05, 0.2, 0.3, 0.5]:
    ablations.append({"model_name": "bert-base-uncased",
                      "train_attention": True,
                      "lambda_attn": 0.3, "ranking_margin": _mv})

In [ ]:
# Seed all randomness for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(config.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(config.seed)
np.random.seed(config.seed)

In [ ]:
from preprocessing import process_and_convert_data
from HateDataset import HateDataset
from torch.utils.data import DataLoader
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from itertools import chain
from bias_evaluation import calculate_gmb_metrics
from HateInterpreter import HateInterpreter
from ExperimentManager import ExperimentManager
from HateClassifier import HateClassifier
import json
import os
import subprocess


def compute_mean_attention_entropy(attentions, test_data, eps=1e-12):
    """Mean attention entropy over the test set (paper Eq. 8).

    `attentions` are CLS, last-layer, head-averaged weights from
    HateClassifier.extract_attention. Aligned to test_data by position
    (predict uses shuffle=False). Restrict to valid (non-pad) tokens,
    renormalize, H = -sum p log p. Same function for every run, so the
    OAT trend is internally consistent.
    """
    H = []
    for att, item in zip(attentions, test_data):
        a = np.asarray(att, dtype=np.float64).flatten()
        mask = item['attention_mask'].cpu().numpy().flatten().astype(bool)
        a = a[mask]
        s = a.sum()
        if s <= 0:
            continue
        p = a / s
        H.append(float(-(p * np.log(p + eps)).sum()))
    return float(np.mean(H)) if H else float('nan')


for ablation in ablations:
  for key, value in ablation.items():
    setattr(config, key, value)

  print(f"\nStarting experiment with config: {ablation}\n")

  # Initialize Experiment Manager
  model_name_safe = config.model_name.replace("/", "-")
  experiment_manager = ExperimentManager(base_dir="./experiments")
  experiment_dir = experiment_manager.create_experiment(
      config=config,
      custom_name=(f"{model_name_safe}_attn{config.train_attention}"
                   f"_lam{config.lambda_attn}_m{config.ranking_margin}")
  )

  # PREPROCESSING
  with open(data_path, 'r') as file:
    data = json.load(file)

  with open('Data/post_id_divisions.json') as file:
      post_id_divisions = json.load(file)

  # Process everything in one pass
  tokenizer = AutoTokenizer.from_pretrained(config.model_name)
  splits = process_and_convert_data(
      data=data,
      tokenizer=tokenizer,
      post_id_divisions=post_id_divisions,
      save_path='Data/explanations/',
      drop_abnormal=False,
      classification_mode=config.classification_mode
  )

  # Access splits directly
  train_data = splits['train']
  val_data = splits['val']
  test_data = splits['test']

  # Create datasets with pre-tokenized data
  train_dataset = HateDataset(data=train_data)
  val_dataset = HateDataset(data=val_data)
  test_dataset = HateDataset(data=test_data)

  train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
  val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False) # Use shuffle=False for validation
  test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False) # Use shuffle=False for testing

  # Compute class weights based on classification mode
  y = [int(td['hard_label']) for td in train_data]
  if config.classification_mode == 'multiclass':
      classes = np.array([0, 1, 2])  # normal, hatespeech, offensive
  else:
      classes = np.array([0, 1])  # normal, toxic

  class_weights = torch.tensor(compute_class_weight(
      class_weight='balanced',
      classes=classes,
      y=y
  ), dtype=torch.float32)

  # TRAINING
  model = HateClassifier(config, class_weight=class_weights)
  history = model.train(train_dataloader=train_loader, val_dataloader=val_loader)

  # Save training history
  experiment_manager.save_training_history(history)

  # EVALUATION
  model.load_model('best_model')

  # Evaluate performance on test set
  result = model.predict(test_dataloader=test_loader, return_attentions=True)

  # Save predictions
  experiment_manager.save_predictions(result, filename="test_predictions.pkl")

  # Mean attention entropy on the test set (paper Eq. 8 / Table 9)
  mean_attention_entropy = compute_mean_attention_entropy(
      result['attentions'], test_data)

  # Evaluate Bias
  all_target_groups = chain.from_iterable([group['target_groups'] for group in train_data])
  all_target_groups = [group for group in all_target_groups if group != 'None' and group != 'Other']
  counter = Counter(all_target_groups)

  n_common = 10
  bias_target_groups = [tg[0] for tg in counter.most_common(n_common)]

  # 4. BIAS EVALUATION
  gmb_metrics, bias_details = calculate_gmb_metrics(
      test_data=test_data,
      probabilities=result['probabilities'],
      target_groups=bias_target_groups,
      classification_mode=config.classification_mode
  )

  # Save bias metrics
  experiment_manager.save_bias_metrics(gmb_metrics, bias_details)

  # 5. XAI EVALUATION (only on hate samples)
  test_data_hate_only = []
  test_results_hate_only = {'attentions': [], 'probabilities': [], 'predictions': [], 'post_id': [], 'labels': []}
  for idx, td in enumerate(test_data):
      if td['hard_label'] == 1 or td['hard_label'] == 2:
          test_data_hate_only.append(td)
          test_results_hate_only['attentions'].append(result['attentions'][idx])
          test_results_hate_only['probabilities'].append(result['probabilities'][idx])
          test_results_hate_only['predictions'].append(result['predictions'][idx])
          test_results_hate_only['post_id'].append(result['post_ids'][idx])
          test_results_hate_only['labels'].append(result['labels'][idx])

  calculator = HateInterpreter(
      model=model,
      tokenizer=tokenizer,
      dataset_class=HateDataset,
      batch_size=32,
      classification_mode=config.classification_mode
  )

  k = 5
  eraser_save_path = f"{experiment_dir}/results/test_explain_output.jsonl"
  xai_results = calculator.compute_all_metrics(test_data_hate_only, test_results_hate_only, k, eraser_save_path)

  # Determine mode parameter for ERASER metrics
  eraser_mode = 'multiclass' if config.classification_mode == 'multiclass' else 'binary'
  score_file = f"{experiment_dir}/metrics/xai_metrics.json"

  # Run ERASER metrics evaluation using subprocess for better variable handling
  env = os.environ.copy()
  env['PYTHONPATH'] = './eraserbenchmark:' + env.get('PYTHONPATH', '')

  subprocess.run([
      'python', 'eraserbenchmark/rationale_benchmark/metrics.py',
      '--split', 'test',
      '--strict',
      '--data_dir', 'Data/explanations',
      '--results', eraser_save_path,
      '--score_file', score_file,
      '--mode', eraser_mode
  ], env=env, check=True)

  with open(score_file) as f:
    xai_results = json.load(f)

  # Final Summary
  # 6. CREATE FINAL SUMMARY
  final_summary = {
      "test_accuracy": float(result['accuracy']),
      "test_f1": float(result['f1']),
      "test_loss": float(result['loss']),
      "attention_entropy": float(mean_attention_entropy),
      "gmb_metrics": gmb_metrics,
      "xai_metrics": xai_results
  }

  experiment_manager.save_final_metrics(final_summary)
  for key, value in final_summary.items():
    if isinstance(value, dict):
        for k, v in value.items():
            print(f"{key}.{k}: {v}")
        print()
    else:
        print(f"{key}: {value}")
    print()

  # 7. MARK EXPERIMENT AS COMPLETE
  experiment_manager.mark_complete(
      status="completed",
      notes="Experiment completed successfully"
  )

  print("\n" + "="*80)
  print("EXPERIMENT COMPLETED!")
  print(f"All results saved to: {experiment_dir}")
  print("="*80)

In [ ]:
# === Hyperparameter sensitivity: aggregate the sweep runs into one CSV ===
# Reads the experiments produced by the run loop above (named
# "<model>_attn<bool>_lam<λ>_m<m>") and builds the λ-sweep / m-sweep table.
import pandas as pd


def _g(d, *path, default=np.nan):
    """Safe nested dict getter."""
    for k in path:
        if isinstance(d, dict) and k in d and d[k] is not None:
            d = d[k]
        else:
            return default
    return d


def _iou_f1(xai):
    # ERASER score_file: scores["iou_scores"] = [{"threshold":0.5,
    #   "micro":{p,r,f1}, "macro":{p,r,f1}}]  (single threshold = 0.5)
    iou = _g(xai, 'iou_scores', default=None)
    if isinstance(iou, list) and iou:
        e = iou[0]
        return _g(e, 'macro', 'f1', default=_g(e, 'micro', 'f1'))
    return np.nan


def extract_row(fs):
    xai = fs.get('xai_metrics', {})
    return {
        'macro_f1':          float(fs.get('test_f1', np.nan)),
        'iou_f1':            float(_iou_f1(xai)),
        'token_f1':          float(_g(xai, 'token_prf', 'instance_macro', 'f1')),
        'auprc':             float(_g(xai, 'token_soft_metrics', 'auprc')),
        'comprehensiveness': float(_g(xai, 'classification_scores', 'comprehensiveness')),
        'sufficiency':       float(_g(xai, 'classification_scores', 'sufficiency')),
        'gmb_bpsn':          float(_g(fs.get('gmb_metrics', {}), 'GMB-BPSN-AUC')),
        'mean_entropy':      float(fs.get('attention_entropy', np.nan)),
    }


def _name(train_attention, lam, m):
    # must match the custom_name built in the run loop (cell with the
    # `for ablation in ablations` loop)
    return f"bert-base-uncased_attn{train_attention}_lam{lam}_m{m}"


# Expected sweep points -> (swept_param, value). The shared center
# (λ=0.3, m=0.1) is reused as both the λ=0.3 and the m=0.1 point.
expected = {}
expected[_name(True, 0.3, 0.1)] = [('lambda', 0.3), ('margin', 0.1)]
for lv in [0.0, 0.1, 0.5, 1.0, 2.0]:
    expected.setdefault(_name(lv > 0, lv, 0.1), []).append(('lambda', float(lv)))
for mv in [0.05, 0.2, 0.3, 0.5]:
    expected.setdefault(_name(True, 0.3, mv), []).append(('margin', float(mv)))

em = ExperimentManager(base_dir="./experiments")
# registry is chronological; keep the latest completed run per name
latest = {}
for e in em.list_experiments(status="completed"):
    if e.get('custom_name') in expected:
        latest[e['custom_name']] = e

rows = []
for cname, mappings in expected.items():
    e = latest.get(cname)
    if e is None:
        print(f"  missing run: {cname}")
        continue
    fp = os.path.join(e['directory'], 'metrics', 'final_summary.json')
    if not os.path.exists(fp):
        print(f"  no final_summary for: {cname}")
        continue
    with open(fp) as f:
        fs = json.load(f)
    metrics = extract_row(fs)
    for swept_param, value in mappings:
        rows.append({'swept_param': swept_param, 'value': value, **metrics})

sens_df = (pd.DataFrame(rows)
           .sort_values(['swept_param', 'value'])
           .reset_index(drop=True))
sens_df.to_csv('sensitivity_results.csv', index=False)
print(sens_df.to_string(index=False))
print('\nSaved: sensitivity_results.csv')

In [ ]:
# === Sensitivity figure: lean 4-curve, 2-panel, camera-ready ===
# Macro-F1 / AUPRC / GMB-BPSN on the primary axis (0-1),
# mean attention entropy on a secondary axis. Reads `sens_df`.
import matplotlib.pyplot as plt

PRIMARY = [('macro_f1', 'Macro-F1', 'tab:blue', 'o', '-'),
           ('auprc', 'AUPRC', 'tab:green', '^', '-'),
           ('gmb_bpsn', 'GMB-BPSN', 'tab:red', 's', '-')]
PANELS = [('lambda', r'$\lambda$  (attention loss weight,  $m=0.1$)', 0.3, True),
          ('margin', r'$m$  (ranking margin,  $\lambda=0.3$)', 0.1, False)]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
leg_h, leg_l = [], []
ent_handle = None

for ax, (param, xlabel, default_x, logx) in zip(axes, PANELS):
    d = sens_df[sens_df.swept_param == param].sort_values('value')
    for col, lbl, color, mk, ls in PRIMARY:
        ln, = ax.plot(d['value'], d[col], color=color, marker=mk,
                      linestyle=ls, linewidth=1.8, markersize=5, label=lbl)
        if lbl not in leg_l:
            leg_h.append(ln)
            leg_l.append(lbl)

    ax2 = ax.twinx()
    ent, = ax2.plot(d['value'], d['mean_entropy'], color='dimgray',
                    marker='D', linestyle='--', linewidth=1.6,
                    markersize=4, label='Attn. entropy')
    ent_handle = ent
    ax2.set_ylim(1.3, 3.05)
    ax2.tick_params(axis='y', colors='dimgray')
    ax2.set_ylabel('Mean attention entropy', color='dimgray')

    ax.axvline(default_x, color='black', linestyle=':', linewidth=1, alpha=0.6)
    ax.annotate('chosen', xy=(default_x, 0.30),
                xytext=(3, 0), textcoords='offset points',
                rotation=90, va='bottom', fontsize=8, color='black', alpha=0.7)

    if logx:
        ax.set_xscale('symlog', linthresh=0.05)
        ax.set_xticks([0, 0.1, 0.3, 0.5, 1.0, 2.0])
        ax.set_xticklabels(['0', '0.1', '0.3', '0.5', '1.0', '2.0'])
    else:
        ax.set_xticks([0.05, 0.1, 0.2, 0.3, 0.5])
    ax.set_xlabel(xlabel)
    ax.set_ylim(0.25, 0.95)
    ax.grid(alpha=0.25)
    ax.set_title('BERT (binary)', fontsize=10)

axes[0].set_ylabel('Metric value')
leg_h.append(ent_handle)
leg_l.append('Attn. entropy')

fig.legend(leg_h, leg_l, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.06), frameon=False)
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig('sensitivity.png', dpi=300, bbox_inches='tight')
fig.savefig('sensitivity.pdf', bbox_inches='tight')
plt.show()
print('Saved: sensitivity.png, sensitivity.pdf')

In [ ]:
# === Appendix table: full per-run numbers (one LaTeX booktabs table) ===
# Two blocks (λ sweep @ m=0.1, m sweep @ λ=0.3) -> sensitivity_table.tex
COLS = ['value', 'macro_f1', 'iou_f1', 'token_f1', 'auprc',
        'comprehensiveness', 'sufficiency', 'gmb_bpsn', 'mean_entropy']
HEAD = (r'$\lambda$/$m$ & Macro-F1 & IOU-F1 & Token-F1 & AUPRC & '
        r'Comp.\ & Suff.\ & GMB-BPSN & Entropy \\')


def _block(param):
    d = sens_df[sens_df.swept_param == param].sort_values('value')
    out = []
    for _, r in d.iterrows():
        cells = [f"{r['value']:g}"] + [f"{r[c]:.3f}" for c in COLS[1:]]
        out.append('  ' + ' & '.join(cells) + r' \\')
    return '\n'.join(out)


latex = r"""\begin{table}[t]
\centering
\caption{Hyperparameter sensitivity of the margin ranking loss (BERT, binary).
One-at-a-time sweeps around the reported operating point ($\lambda=0.3$,
$m=0.1$); $\lambda=0$ is the no-supervision baseline.}
\label{tab:sensitivity}
\begin{tabular}{lcccccccc}
\toprule
""" + HEAD + r"""
\midrule
\multicolumn{9}{l}{\textit{$\lambda$ sweep ($m=0.1$)}} \\
""" + _block('lambda') + r"""
\midrule
\multicolumn{9}{l}{\textit{$m$ sweep ($\lambda=0.3$)}} \\
""" + _block('margin') + r"""
\bottomrule
\end{tabular}
\end{table}
"""

with open('sensitivity_table.tex', 'w', encoding='utf-8') as f:
    f.write(latex)
print(latex)
print('Saved: sensitivity_table.tex')

## Experiment Management Utilities

Useful commands for managing and comparing experiments:

In [ ]:
# View all experiments
experiment_manager.print_experiment_summary()

In [ ]:
import os
import shutil

# List only completed experiments
completed_experiments = experiment_manager.list_experiments(status="completed")
print(f"Found {len(completed_experiments)} completed experiments")

# Create parent folder for all zips
parent_folder = "all_experiment_metrics"
if os.path.exists(parent_folder):
    shutil.rmtree(parent_folder)  # Remove if exists
os.makedirs(parent_folder)

for exp in completed_experiments:
    print(f"  - {exp['experiment_id']}: {exp.get('description', 'No description')}")
    current_folder = f"{parent_folder}/{exp['experiment_id']}"
    os.makedirs(current_folder)
    
    # Create individual zip inside parent folder
    zip_name = f"{current_folder}/metrics"
    !zip -r {zip_name}.zip experiments/{exp['experiment_id']}/metrics/

    zip_result = f"{current_folder}/results"
    !zip -r {zip_result}.zip experiments/{exp['experiment_id']}/results/

# Zip the entire parent folder
!zip -r all_experiment_metrics.zip {parent_folder}/

print(f"\nAll individual zips saved in: {parent_folder}/")
print(f"Final compressed archive: all_experiment_metrics.zip")

In [ ]:
# Compare multiple experiments
experiment_ids = [exp['experiment_id'] for exp in completed_experiments]
comparison = experiment_manager.compare_experiments(experiment_ids)

# Display comparison
for exp in comparison["experiments"]:
    print(f"\nExperiment: {exp['experiment_id']}")
    print(f"  Model: {exp['config'].get('model_name', 'N/A')}")
    print(f"  Learning Rate: {exp['config'].get('learning_rate', 'N/A')}")
    print(f"  Test F1: {exp['metrics'].get('test_f1', 'N/A')}")
    print(f"  Test Accuracy: {exp['metrics'].get('test_accuracy', 'N/A')}")

## Visualization Tools

Visualize and compare experiment results:

In [ ]:
from experiment_visualization import (
    plot_training_curves,
    plot_metrics_comparison,
    plot_bias_metrics,
    plot_xai_metrics,
    create_experiment_report
)

In [ ]:
# Uncomment to use:
plot_training_curves(experiment_ids, save_path="training_curves.png")

In [ ]:
# Compare final metrics across experiments
plot_metrics_comparison(experiment_ids, save_path="metrics_comparison.png")

In [ ]:
# Visualize bias metrics
plot_bias_metrics(experiment_ids, save_path="bias_comparison.png")

In [ ]:
# Visualize XAI metrics
plot_xai_metrics(experiment_ids, save_path="xai_comparison.png")